# Raw data to cropped images (BIN -> 256*256 pixels png)

In [1]:
import gzip
import netCDF4 as nc
import numpy as np
import os
import tarfile

from glob import glob
from PIL import Image
from tqdm import tqdm

# Adding the parent directory to the system path so that we can import modules from there
import sys
sys.path.append('../')

from utils import *

## Extracting tar.gz file

In [2]:
dst_dir = '/home/u2018144071/사기설/HSR/'

input_names = np.array(sorted(glob(dst_dir+'*.tar.gz')))

for i in input_names:
    path = i[:-7]
    with tarfile.open(i, 'r:gz') as tr:
        tr.extractall(path=path)

## Image cropping

In [37]:
def list_files(src_dir, file_format):
    """
    Retrieve a sorted array of file paths in a directory and its subdirectories, filtered by a specified file format.

    Parameters
    ----------
    src_dir : str
        The source directory from which to start searching for files.
    file_format : str
        The desired file format or file extension to filter the files.

    Returns
    -------
    file_names : np.ndarray
        A sorted NumPy array containing the full paths of files that match the specified file format. (in this case, '.bin.gz')
    """
    # Initialize a list to store file paths
    file_names = []

    for (root, directories, files) in os.walk(src_dir):
        for file in files:
            if file.endswith(file_format):
                # Create the full path of the file and add it to the list
                img_dir = os.path.join(root, file)
                file_names.append(img_dir)

    # Sort the file paths in chronological order and convert them into a NumPy array
    file_names = np.array(sorted(file_names))

    return file_names


def img_cropping_and_saving(src_dir, dst_dir, target_idx, size=256):
    """
    Crop and save the KMA radar reflectivity images from a specified source directory to a destination directory.

    Parameters
    ----------
    src_dir : str
        The source directory containing input images in binary gzip format.
    dst_dir : str
        The destination directory where cropped and converted images will be saved.
    target_idx : tuple
        A tuple representing the target indices (row, column) for cropping around the region of interest.
        (in this study, for seoul area, (1440, 1000))
    size : int, optional
        The size of the cropped images. In this study default is 256.

    Returns
    -------
    error_names : list
        A list of file paths for which errors occurred during processing.
        In the future, these data will be processed separately.
    """
    file_format = '.bin.gz'
    input_names = list_files(src_dir, file_format)
    n_row_target, n_col_target = target_idx

    error_names = []
    L = len(input_names)
    print(f'Image cropping is started\nWe have {L} images to be cropped')

    for idx, gz_path in tqdm(enumerate(input_names), total=L, ncols=100, dynamic_ncols=True):
        try:
            data = gzip.open(gz_path).read()
        except Exception as e:
            print(f"An error occurred for {gz_path}: {e}")
            error_names.append(gz_path)
            continue

        dBZ = np.frombuffer(data[1024:], dtype=np.int16) / 100.0
        dBZ = dBZ.reshape((2881, 2305))

        # QC Reflectivity" images that are less than 0 are not typically observed physically.
        # Therefore, it is deemed that these images were incorrectly measured, and they are replaced with 0.0.
        dBZ[dBZ <= 0.0] = 0.0

        # Flip the image upside down (coordinate system is flipped)
        dBZ = np.flipud(dBZ)  

        # Image cropping : Crop the region of interest (ROI) with a specified size around the target indices.
        d = int(size / 2)
        dBZ_cropped = dBZ[n_row_target - d: n_row_target + d, n_col_target - d: n_col_target + d]

        # Convert to grayscale (0–255)
        img_ = dBZ_to_grayscale(dBZ_cropped)
        img = Image.fromarray(img_.astype(np.uint8))

        # Save as a PNG file
        img_dir = os.path.join(dst_dir, gz_path[-19:-13], gz_path[-13:-11])
        os.makedirs(img_dir, exist_ok=True)
        img.save(os.path.join(img_dir, gz_path[-19:-7] + '.png'))

    print('Image Cropping is finished')

    return error_names

In [38]:
src_dir = '/home/u2018144071/사기설/HSR'
dst_dir = '/home/u2018144071/사기설/data/cropped'
target_idx = (1299, 1295)

error_names = img_cropping_and_saving(src_dir, dst_dir, target_idx)

We have 394216 images for training!
Image cropping is started
We have 137682 images to be cropped
An error occurred for /home/u2018144071/사기설/HSR/RDR_HSR_202206/09/RDR_CMP_HSR_PUB_202206092005.bin.gz: CRC check failed 0xcca44147 != 0x53a9f75d
1000/137682 images are cropped...
An error occurred for /home/u2018144071/사기설/HSR/RDR_HSR_202206/15/RDR_CMP_HSR_PUB_202206152015.bin.gz: Not a gzipped file (b'\x00\x00')
2000/137682 images are cropped...
An error occurred for /home/u2018144071/사기설/HSR/RDR_HSR_202206/19/RDR_CMP_HSR_PUB_202206190300.bin.gz: Error -3 while decompressing data: invalid literal/length code
An error occurred for /home/u2018144071/사기설/HSR/RDR_HSR_202206/19/RDR_CMP_HSR_PUB_202206192110.bin.gz: Not a gzipped file (b'\x00\x00')
3000/137682 images are cropped...
4000/137682 images are cropped...
5000/137682 images are cropped...
6000/137682 images are cropped...
7000/137682 images are cropped...
8000/137682 images are cropped...
9000/137682 images are cropped...
10000/137682 

## Exception handling

In [ ]:
# Code for handling 5 images that raised errors: Processing .nc files separately.

for gz_path in range(error_names):
    nc_path = f'/home/u2018144071/사기설/error/RDR_CMP_HSR_PUB_skorea_{gz_path[-19:-7]}.nc'

    # Target indices for cropping (.nc file and .bin.gz file have different target_idx)
    target_idx = (1022, 718)
    n_col_target, n_row_target = target_idx
    size = 256

    data = nc.Dataset(nc_path)
    dBZ = data.variables['value'][:][0,...]

    # QC Reflectivity" images that are less than 0 are not typically observed physically.
    # Therefore, it is deemed that these images were incorrectly measured, and they are replaced with 0.0.
    dBZ[dBZ <= 0.0] = 0.0

    # Flip the image upside down (coordinate system is flipped)
    dBZ = np.flipud(dBZ)  

    # Image cropping : Crop the region of interest (ROI) with a specified size around the target indices.
    d = int(size / 2)
    dBZ_cropped = dBZ[n_row_target - d: n_row_target + d, n_col_target - d: n_col_target + d]

    # Convert to grayscale (0–255)
    img_ = dBZ_to_grayscale(dBZ_cropped)
    img = Image.fromarray(img_.astype(np.uint8))

    # Save as a PNG file
    img_dir = os.path.join(dst_dir, nc_path[-15:-9], nc_path[-9:-7])
    os.makedirs(img_dir, exist_ok=True)
    img.save(os.path.join(img_dir, nc_path[-15:-3] + '.png'))
    
    print(f'Image is saved to {os.path.join(img_dir, nc_path[-15:-3] + ".png")}')